In [2]:
# =============================================================================
# NOTEBOOK: wf_environment (v1.0.2)
# PROJECT:  Western Flyer - Sea of Cortez CTD Research Suite (2025)
# PURPOSE:  Environment Audit, Directory Setup, and DuckDB Schema Initialization
# UPDATES:  All third‑party imports moved below audit_environment().
# =============================================================================

# --- 0. SAFE IMPORTS (Standard Library Only) ---
import os
import sys
import subprocess
import pathlib

# --- 1. ENVIRONMENT AUDIT & AUTO-INSTALLER ---
# Ensures that both the Master Notebook (Math/Processing)
# and the Research Suite (Visualization/Dashboard) have all dependencies.

REQUIRED_PACKAGES = [
    'duckdb',           # Database Engine
    'pandas',           # Data Structures
    'numpy',            # Numerical Arrays
    'gsw',              # TEOS-10 Thermodynamic Engine
    'scipy',            # Signal Processing (Savgol Filter)
    'holoviews',        # Reactive Visualizations
    'panel',            # Dashboard Framework
    'geoviews',         # Geographic Data Visualization
    'cartopy',          # Geospatial Projections
    'watchfiles'        # Required for Panel --autoreload stability
]

def audit_environment():
    """Checks for required packages and installs missing ones via pip."""
    print("🔍 Auditing Environment for Sea of Cortez Research Suite...")
    for pkg in REQUIRED_PACKAGES:
        try:
            __import__(pkg)
        except ImportError:
            print(f"📦 Missing: {pkg}. Installing now...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    print("✅ Environment Audit Complete: All dependencies verified.")

# Run the audit BEFORE importing any third‑party packages
audit_environment()

# --- 2. THIRD‑PARTY IMPORTS (Safe After Audit) ---
import duckdb
import pandas as pd
import numpy as np
import gsw
import scipy
import holoviews as hv
import panel as pn
import geoviews as gv
import cartopy
import watchfiles

# --- 3. DIRECTORY ARCHITECTURE ---
# Create the local 'processed' folder if it does not exist
pathlib.Path("processed").mkdir(exist_ok=True)
DATABASE_PATH = "processed/sea_of_cortez.duckdb"

# --- 4. DUCKDB SCHEMA INITIALIZATION ---
# Ensures the Build Notebook and Dashboard App use the same schema.

with duckdb.connect(DATABASE_PATH) as con:
    print(f"\n📡 Connecting to Database: {DATABASE_PATH}")

    # Resetting the table to ensure a clean start with current standards
    con.execute("DROP TABLE IF EXISTS wf_ctd_binned")

    con.execute("""
        CREATE TABLE wf_ctd_binned (
            station_id VARCHAR,      -- Unique ID (e.g., PuntaMarciel_C4)
            station_name VARCHAR,    -- Descriptive Location
            cast_no INTEGER,         -- Sequence of the cast
            dbar_bin DOUBLE,         -- Pressure/Depth Bin (0.1 res)
            time_iso TIMESTAMP,      -- ISO 8601 Timestamp
            time_elapsed DOUBLE,     -- Seconds from start of cast
            CT DOUBLE,               -- Conservative Temperature (°C)
            SA DOUBLE,               -- Absolute Salinity (g/kg)
            SP DOUBLE,               -- Practical Salinity (PSU)
            rho DOUBLE,              -- In-situ Density (kg/m3)
            o2_final DOUBLE,         -- Dissolved Oxygen (umol/kg)
            ph_final DOUBLE,         -- pH (Total Scale)
            chl_final DOUBLE,        -- Chlorophyll-a (mg/m3)
            sigma0 DOUBLE,           -- Potential Density (kg/m3)
            n2 DOUBLE,               -- Stability (Brunt-Vaisala Frequency)
            spice DOUBLE,            -- Seawater Spiciness
            sound_speed DOUBLE,      -- Speed of Sound (m/s)
            lat DOUBLE,              -- Decimal Latitude
            lon DOUBLE,              -- Decimal Longitude
            depth_m DOUBLE,          -- Vertical Coordinate (Positive Meters)
            dp DOUBLE,               -- Pressure Delta (Descent Rate Audit)
            qc_flag INTEGER,         -- Quality Control (1=Pass, 3=Suspect, 4=Fail)
            pipeline_version VARCHAR -- Version Tracking for Data Integrity
        )
    """)

    # Verification Query to ensure all columns are present
    table_check = con.execute("SELECT * FROM wf_ctd_binned LIMIT 0").df()

    print(f"✅ Infrastructure Ready.")
    print(f"🚀 Initialized table 'wf_ctd_binned' with {len(table_check.columns)} scientific columns.")
    print(f"📊 Schema v1.1.0 is now compatible with wf_ctd_build and Dashboard App.")

🔍 Auditing Environment for Sea of Cortez Research Suite...
📦 Missing: duckdb. Installing now...
📦 Missing: gsw. Installing now...
📦 Missing: geoviews. Installing now...
📦 Missing: watchfiles. Installing now...
✅ Environment Audit Complete: All dependencies verified.

📡 Connecting to Database: processed/sea_of_cortez.duckdb
✅ Infrastructure Ready.
🚀 Initialized table 'wf_ctd_binned' with 23 scientific columns.
📊 Schema v1.1.0 is now compatible with wf_ctd_build and Dashboard App.
